# Tomahawk Cassette Rev3.1 — Controlled Local-Deposition Envelope

This notebook evaluates one external CSV dataset and rejects any file whose `analysis_branch` does not equal `transport_deposition_controlled`. It automatically discovers renamed Kaggle attachments by schema and branch signature, so the CSV does not need to retain its original filename.

The thermal-hydraulic equations, constants, and gates are unchanged from the combined Rev3.1 evaluator. The result is deterministic coverage of the declared input grid—not a probability of hardware success and not CFD, neutron transport, structural, irradiation, or experimental validation.

Acceptance rule: at least 60% of the declared deterministic grid passes all gates. No GPU is required.


In [1]:
# SECTION 1 — imports, deterministic configuration, and output paths
import hashlib
import json
import math
import platform
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 20260826
np.random.seed(SEED)
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'tomahawk_cassette_rev3_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROLLED_NAME = 'Tomahawk_Cassette_Rev3_Controlled_Envelope_Input.csv'
ENGINEERED_NAME = 'Tomahawk_Cassette_Rev3_Engineered_Full_Incident_Input.csv'
STRESS_NAME = 'Tomahawk_Cassette_Rev3_Full_Incident_Stress_Input.csv'

CONSTANTS = {
    'n_cassettes_per_module': 4,
    'cassette_area_m2': 2.5,
    'rf_power_absorbed_MW_module': 15.0,
    'rho_kg_m3': 950.0,
    'mu_Pa_s': 2.30e-4,
    'water_k_W_mK': 0.68,
    'cp_J_kgK': 4180.0,
    'cucrzr_k_W_mK': 300.0,
    'minor_loss_K': 3.0,
    'pump_efficiency': 0.70,
}

GATES = {
    'max_CuCrZr_temperature_C': 300.0,
    'minimum_boiling_margin_C': 10.0,
    'max_velocity_m_s': 5.0,
    'max_pressure_drop_MPa': 0.25,
    'max_pump_power_MW_module': 1.0,
    'max_flow_area_fraction': 0.08,
}

TOPOLOGY = {
    'parallel_smooth': {'Nu_multiplier': 1.00, 'dp_multiplier': 1.00},
    'parallel_ribbed': {'Nu_multiplier': 1.25, 'dp_multiplier': 1.60},
}

print('Python:', platform.python_version())
print('Output:', OUT_DIR)
print('Seed:', SEED)
print('GPU required: no')

EXPECTED_BRANCH = 'transport_deposition_controlled'
PREFERRED_INPUT_NAME = 'Tomahawk_Cassette_Rev3_Controlled_Envelope_Input.csv'
RUN_SLUG = 'controlled_deposition'
FROZEN_EVALUATOR_SOURCE_SHA256 = '8e509d0f5ac31fdbef099e14c54324632a72c4495e6895ed5de4f3eea46e4f09'

print('Expected analysis branch:', EXPECTED_BRANCH)
print('Preferred input filename:', PREFERRED_INPUT_NAME)
print('Frozen evaluator source SHA-256:', FROZEN_EVALUATOR_SOURCE_SHA256)


Python: 3.12.13
Output: /kaggle/working
Seed: 20260826
GPU required: no
Expected analysis branch: transport_deposition_controlled
Preferred input filename: Tomahawk_Cassette_Rev3_Controlled_Envelope_Input.csv
Frozen evaluator source SHA-256: 8e509d0f5ac31fdbef099e14c54324632a72c4495e6895ed5de4f3eea46e4f09


In [2]:
# SECTION 2 — discover one external input by filename or branch signature
required = [
    'analysis_branch', 'topology', 'q_nuclear_areal_MW_m2', 'rf_loss_fraction',
    'pressure_MPa', 'inlet_temp_C', 'deltaT_C', 'channel_count_per_cassette',
    'Dh_mm', 'cucrzr_skin_mm', 'channel_length_m'
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def candidate_csv_paths():
    paths = list(Path.cwd().glob('*.csv'))
    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        paths.extend(kaggle_root.rglob('*.csv'))
    return sorted({path.resolve() for path in paths if path.is_file()}, key=str)

candidates = candidate_csv_paths()
if not candidates:
    raise FileNotFoundError(
        'No CSV was found. Attach the one branch input dataset through Kaggle Add Input, then Run All.'
    )

matches = []
rejections = []
for path in candidates:
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        rejections.append((str(path), f'unreadable: {type(exc).__name__}'))
        continue
    missing = [column for column in required if column not in frame.columns]
    if missing:
        rejections.append((str(path), 'schema mismatch'))
        continue
    branches = set(frame['analysis_branch'].dropna().astype(str).unique())
    if branches != {EXPECTED_BRANCH}:
        rejections.append((str(path), f'branch mismatch: {sorted(branches)}'))
        continue
    matches.append((path, frame))

if not matches:
    inventory = '\n'.join(f'  - {path}: {reason}' for path, reason in rejections)
    raise FileNotFoundError(
        f'No CSV matched expected branch {EXPECTED_BRANCH!r}. Discovered files:\n{inventory}'
    )

preferred_matches = [item for item in matches if item[0].name.casefold() == PREFERRED_INPUT_NAME.casefold()]
selection_pool = preferred_matches or matches
hashes = {sha256(path) for path, _ in selection_pool}
if len(selection_pool) > 1 and len(hashes) > 1:
    paths = '\n'.join(f'  - {path}' for path, _ in selection_pool)
    raise RuntimeError(f'Multiple nonidentical datasets match {EXPECTED_BRANCH!r}:\n{paths}')

input_path, inputs = selection_pool[0]
if inputs.empty:
    raise ValueError('The selected input dataset has zero rows.')
if not np.isfinite(inputs[required[2:]].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite numeric input found.')
if not set(inputs['topology']).issubset(TOPOLOGY):
    raise ValueError('Unknown topology found in input data.')

print('Selected input:', input_path)
print('Rows:', len(inputs))
print('Branch:', EXPECTED_BRANCH)
print('Input SHA-256:', sha256(input_path))
print('Input validation: PASS')


Selected input: /kaggle/input/datasets/nicolebryant29/controlled/Tomahawk_Cassette_Rev3_Controlled_Envelope_Input (1).csv
Rows: 1728
Branch: transport_deposition_controlled
Input SHA-256: ad7edd9e93eb8a1e3d6d6d5f3897e952b47a60b6cadd82355b19c126ec03128f
Input validation: PASS


In [3]:
# SECTION 3 — vectorized hydraulic and thermal evaluator
def saturation_temperature_C(p_MPa):
    return np.interp(np.asarray(p_MPa, dtype=float), [3, 4, 5, 6, 7], [233.9, 250.4, 263.9, 275.6, 285.8])

def evaluate(frame):
    x = frame.copy()
    c = CONSTANTS
    total_area = c['n_cassettes_per_module'] * c['cassette_area_m2']
    q_nuclear_W = x['q_nuclear_areal_MW_m2'].to_numpy(float) * 1e6 * total_area
    q_rf_W = c['rf_power_absorbed_MW_module'] * 1e6 * x['rf_loss_fraction'].to_numpy(float)
    q_total_W = q_nuclear_W + q_rf_W
    dT = x['deltaT_C'].to_numpy(float)
    mdot_module = q_total_W / (c['cp_J_kgK'] * dT)
    mdot_cassette = mdot_module / c['n_cassettes_per_module']

    Dh = x['Dh_mm'].to_numpy(float) / 1000.0
    n_channels = x['channel_count_per_cassette'].to_numpy(float)
    flow_area = n_channels * Dh ** 2
    velocity = mdot_cassette / (c['rho_kg_m3'] * flow_area)
    Re = c['rho_kg_m3'] * velocity * Dh / c['mu_Pa_s']
    Pr = c['cp_J_kgK'] * c['mu_Pa_s'] / c['water_k_W_mK']
    Nu_base = np.where(Re < 2300.0, 4.36, 0.023 * Re ** 0.8 * Pr ** 0.4)
    nu_mult = x['topology'].map({k:v['Nu_multiplier'] for k,v in TOPOLOGY.items()}).to_numpy(float)
    dp_mult = x['topology'].map({k:v['dp_multiplier'] for k,v in TOPOLOGY.items()}).to_numpy(float)
    Nu = Nu_base * nu_mult
    h = Nu * c['water_k_W_mK'] / Dh

    friction_base = np.where(Re < 2300.0, 64.0 / np.maximum(Re, 1.0), 0.3164 / Re ** 0.25)
    dynamic_pressure = 0.5 * c['rho_kg_m3'] * velocity ** 2
    length = x['channel_length_m'].to_numpy(float)
    dp_Pa = (friction_base * dp_mult * length / Dh + c['minor_loss_K'] * dp_mult) * dynamic_pressure

    q_face_W_m2 = q_total_W / total_area
    skin = x['cucrzr_skin_mm'].to_numpy(float) / 1000.0
    wall_temp = (
        x['inlet_temp_C'].to_numpy(float) + dT
        + q_face_W_m2 / h
        + q_face_W_m2 * skin / c['cucrzr_k_W_mK']
    )
    Tsat = saturation_temperature_C(x['pressure_MPa'])
    pump_power_W = dp_Pa * (mdot_module / c['rho_kg_m3']) / c['pump_efficiency']
    flow_area_fraction = flow_area / c['cassette_area_m2']

    x['q_total_MW_module'] = q_total_W / 1e6
    x['mdot_kg_s_module'] = mdot_module
    x['flow_area_m2_cassette'] = flow_area
    x['flow_area_fraction'] = flow_area_fraction
    x['velocity_m_s'] = velocity
    x['Re'] = Re
    x['Nu'] = Nu
    x['h_W_m2K'] = h
    x['pressure_drop_kPa'] = dp_Pa / 1000.0
    x['pump_power_MW_module'] = pump_power_W / 1e6
    x['wall_temp_C'] = wall_temp
    x['saturation_temp_C'] = Tsat
    x['boiling_margin_C'] = Tsat - wall_temp
    x['thermal_margin_C'] = GATES['max_CuCrZr_temperature_C'] - wall_temp

    x['gate_temperature'] = wall_temp <= GATES['max_CuCrZr_temperature_C']
    x['gate_boiling_margin'] = (Tsat - wall_temp) >= GATES['minimum_boiling_margin_C']
    x['gate_velocity'] = velocity <= GATES['max_velocity_m_s']
    x['gate_pressure_drop'] = dp_Pa <= GATES['max_pressure_drop_MPa'] * 1e6
    x['gate_pump_power'] = pump_power_W <= GATES['max_pump_power_MW_module'] * 1e6
    x['gate_flow_area'] = flow_area_fraction <= GATES['max_flow_area_fraction']
    gate_cols = [c for c in x if c.startswith('gate_')]
    x['passes_all_gates'] = x[gate_cols].all(axis=1)
    return x

start = time.perf_counter()
results = evaluate(inputs)
elapsed = time.perf_counter() - start
print('Evaluated rows:', len(results), 'elapsed_seconds:', round(elapsed, 4))

Evaluated rows: 1728 elapsed_seconds: 0.0127


In [4]:
# SECTION 4 — single-branch results and acceptance check
gate_cols = [column for column in results if column.startswith('gate_')]
summary = pd.DataFrame([{
    'analysis_branch': EXPECTED_BRANCH,
    'rows': int(len(results)),
    'passing_rows': int(results['passes_all_gates'].sum()),
    'pass_percent': float(100.0 * results['passes_all_gates'].mean()),
    'minimum_wall_temp_C': float(results['wall_temp_C'].min()),
    'maximum_wall_temp_C': float(results['wall_temp_C'].max()),
    'minimum_boiling_margin_C': float(results['boiling_margin_C'].min()),
    'maximum_velocity_m_s': float(results['velocity_m_s'].max()),
    'maximum_pressure_drop_kPa': float(results['pressure_drop_kPa'].max()),
    'maximum_pump_power_MW': float(results['pump_power_MW_module'].max()),
    'minimum_passing_mass_flow_kg_s_module': (
        float(results.loc[results['passes_all_gates'], 'mdot_kg_s_module'].min())
        if results['passes_all_gates'].any() else np.nan
    ),
    'maximum_passing_mass_flow_kg_s_module': (
        float(results.loc[results['passes_all_gates'], 'mdot_kg_s_module'].max())
        if results['passes_all_gates'].any() else np.nan
    ),
}])
gate_summary = pd.DataFrame({
    'gate': gate_cols,
    'pass_percent': [float(100.0 * results[column].mean()) for column in gate_cols],
})

display(summary.round(4))
display(gate_summary.round(2))

pass_percent = float(summary.loc[0, 'pass_percent'])
passing_rows = int(summary.loc[0, 'passing_rows'])
if EXPECTED_BRANCH in {'transport_deposition_controlled', 'full_incident_engineered'}:
    acceptance_passed = pass_percent >= 60.0
    acceptance_rule = 'pass_percent >= 60.0'
else:
    acceptance_passed = passing_rows > 0
    acceptance_rule = 'passing_rows > 0'

print('ACCEPTANCE RULE:', acceptance_rule)
print('ACCEPTANCE CHECK:', 'PASS' if acceptance_passed else 'FAIL')
print('Interpretation: deterministic grid coverage only; not a hardware-success probability.')
if EXPECTED_BRANCH == 'full_incident_engineered':
    print('System-integration caution: minimum passing mass flow =',
          round(float(summary.loc[0, 'minimum_passing_mass_flow_kg_s_module']), 2),
          'kg/s/module; no plant-level mass-flow feasibility gate has been established.')

if not acceptance_passed:
    raise RuntimeError(f'Acceptance check failed: {acceptance_rule}.')


,analysis_branch,rows,passing_rows,pass_percent,minimum_wall_temp_C,maximum_wall_temp_C,minimum_boiling_margin_C,maximum_velocity_m_s,maximum_pressure_drop_kPa,maximum_pump_power_MW,minimum_passing_mass_flow_kg_s_module,maximum_passing_mass_flow_kg_s_module
0,transport_deposition_controlled,1728,1346,77.8935,157.8247,347.4344,-83.5344,2.4208,17.7654,0.003,54.2265,113.0383


,gate,pass_percent
0,gate_temperature,95.14
1,gate_boiling_margin,77.89
2,gate_velocity,100.00
3,gate_pressure_drop,100.00
4,gate_pump_power,100.00
5,gate_flow_area,100.00


ACCEPTANCE RULE: pass_percent >= 60.0
ACCEPTANCE CHECK: PASS
Interpretation: deterministic grid coverage only; not a hardware-success probability.


In [5]:
# SECTION 5 — rank candidates, plot, and export a branch-specific result bundle
feasible = results[results['passes_all_gates']].copy()
feasible['ranking_score'] = (
    feasible['thermal_margin_C']
    + 0.20 * feasible['boiling_margin_C']
    - 0.08 * feasible['pressure_drop_kPa']
    - 5.0 * feasible['pump_power_MW_module']
    - 2.0 * np.maximum(feasible['velocity_m_s'] - 3.0, 0.0)
    - 0.05 * feasible['mdot_kg_s_module']
)
top = feasible.sort_values('ranking_score', ascending=False).head(25)

report_columns = [
    'topology', 'q_nuclear_areal_MW_m2', 'rf_loss_fraction', 'pressure_MPa',
    'inlet_temp_C', 'deltaT_C', 'channel_count_per_cassette', 'Dh_mm',
    'cucrzr_skin_mm', 'channel_length_m', 'mdot_kg_s_module', 'velocity_m_s',
    'pressure_drop_kPa', 'pump_power_MW_module', 'wall_temp_C',
    'boiling_margin_C', 'ranking_score'
]
print('Top passing candidates:')
display(top[report_columns].round(4))

results_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_All_Results.csv'
summary_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Summary.csv'
gate_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Gate_Rates.csv'
top_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Top_Candidates.csv'
plot_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Results.png'
manifest_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Manifest.json'
zip_path = OUT_DIR / f'Tomahawk_Cassette_Rev3_1_{RUN_SLUG}_Result_Bundle.zip'

results.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)
gate_summary.to_csv(gate_path, index=False)
top.to_csv(top_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = np.where(results['passes_all_gates'], '#2E7D32', '#B71C1C')
axes[0].scatter(results['pressure_drop_kPa'], results['wall_temp_C'], c=colors, s=8, alpha=.35)
axes[0].axhline(GATES['max_CuCrZr_temperature_C'], color='black', linestyle='--')
axes[0].set(xlabel='Pressure drop [kPa]', ylabel='Wall temperature [°C]', title='Thermal-hydraulic grid')
axes[0].grid(alpha=.2)
axes[1].bar(gate_summary['gate'].str.replace('gate_', '', regex=False), gate_summary['pass_percent'], color='#2E74B5')
axes[1].set(ylabel='Gate pass rate [%]', title='Individual gate coverage', ylim=(0, 100))
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y', alpha=.2)
plt.tight_layout()
plt.savefig(plot_path, dpi=170)
plt.close()

manifest = {
    'model': 'Tomahawk Cassette Rev3.1 single-branch reduced-order thermal-hydraulic screen',
    'analysis_branch': EXPECTED_BRANCH,
    'scope': 'deterministic reduced-order screen; not CFD, neutronics, structural, irradiation, or hardware validation',
    'input_path': str(input_path),
    'input_sha256': sha256(input_path),
    'frozen_evaluator_source_sha256': FROZEN_EVALUATOR_SOURCE_SHA256,
    'constants': CONSTANTS,
    'gates': GATES,
    'summary': summary.to_dict(orient='records')[0],
    'acceptance_rule': acceptance_rule,
    'acceptance_passed': bool(acceptance_passed),
    'elapsed_seconds': float(elapsed),
}
manifest_path.write_text(json.dumps(manifest, indent=2))

files = [results_path, summary_path, gate_path, top_path, plot_path, manifest_path]
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.name)

print('Saved result bundle:', zip_path)
print(json.dumps(manifest, indent=2))


Top passing candidates:


,topology,q_nuclear_areal_MW_m2,rf_loss_fraction,pressure_MPa,inlet_temp_C,deltaT_C,channel_count_per_cassette,Dh_mm,cucrzr_skin_mm,channel_length_m,mdot_kg_s_module,velocity_m_s,pressure_drop_kPa,pump_power_MW_module,wall_temp_C,boiling_margin_C,ranking_score
864,parallel_ribbed,0.650,0.02,5.0,90.0,20.0,48,16.0,2.0,0.75,81.3397,1.7420,8.7753,0.0011,157.8247,106.0753,158.6160
865,parallel_ribbed,0.650,0.02,5.0,90.0,20.0,48,16.0,2.0,1.00,81.3397,1.7420,9.3943,0.0011,157.8247,106.0753,158.5661
1008,parallel_ribbed,0.650,0.03,5.0,90.0,20.0,48,16.0,2.0,0.75,83.1340,1.7804,9.1562,0.0011,158.1140,105.7860,158.1483
1009,parallel_ribbed,0.650,0.03,5.0,90.0,20.0,48,16.0,2.0,1.00,83.1340,1.7804,9.7992,0.0012,158.1140,105.7860,158.0964
866,parallel_ribbed,0.650,0.02,5.0,90.0,20.0,48,16.0,3.0,0.75,81.3397,1.7420,8.7753,0.0011,160.0914,103.8086,155.8960
867,parallel_ribbed,0.650,0.02,5.0,90.0,20.0,48,16.0,3.0,1.00,81.3397,1.7420,9.3943,0.0011,160.0914,103.8086,155.8461
1010,parallel_ribbed,0.650,0.03,5.0,90.0,20.0,48,16.0,3.0,0.75,83.1340,1.7804,9.1562,0.0011,160.4307,103.4693,155.3683
1011,parallel_ribbed,0.650,0.03,5.0,90.0,20.0,48,16.0,3.0,1.00,83.1340,1.7804,9.7992,0.0012,160.4307,103.4693,155.3164
1152,parallel_ribbed,0.772,0.02,5.0,90.0,20.0,48,16.0,2.0,0.75,95.9330,2.0545,12.1022,0.0017,160.0906,103.8094,154.8977
1153,parallel_ribbed,0.772,0.02,5.0,90.0,20.0,48,16.0,2.0,1.00,95.9330,2.0545,12.9284,0.0019,160.0906,103.8094,154.8310


Saved result bundle: /kaggle/working/Tomahawk_Cassette_Rev3_1_controlled_deposition_Result_Bundle.zip
{
  "model": "Tomahawk Cassette Rev3.1 single-branch reduced-order thermal-hydraulic screen",
  "analysis_branch": "transport_deposition_controlled",
  "scope": "deterministic reduced-order screen; not CFD, neutronics, structural, irradiation, or hardware validation",
  "input_path": "/kaggle/input/datasets/nicolebryant29/controlled/Tomahawk_Cassette_Rev3_Controlled_Envelope_Input (1).csv",
  "input_sha256": "ad7edd9e93eb8a1e3d6d6d5f3897e952b47a60b6cadd82355b19c126ec03128f",
  "frozen_evaluator_source_sha256": "8e509d0f5ac31fdbef099e14c54324632a72c4495e6895ed5de4f3eea46e4f09",
  "constants": {
    "n_cassettes_per_module": 4,
    "cassette_area_m2": 2.5,
    "rf_power_absorbed_MW_module": 15.0,
    "rho_kg_m3": 950.0,
    "mu_Pa_s": 0.00023,
    "water_k_W_mK": 0.68,
    "cp_J_kgK": 4180.0,
    "cucrzr_k_W_mK": 300.0,
    "minor_loss_K": 3.0,
    "pump_efficiency": 0.7
  },
  "gates": 